# Member 1 — Compact CUB-200-2011 Milestone 1

**Data loading · EDA · image review · caption cleaning · stratified split · leak prevention · transforms · full DataLoader verification.**

Run from the **project root** or `notebooks/`. The notebook uses helper functions in `src/data/member1_pipeline.py` and the reusable training interface in `src/data/cub_data.py`. These files are required. Existing image-review decisions are preserved.

Install packages first: `pip install datasets pillow matplotlib pandas numpy tqdm scikit-learn torch torchvision`.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# Open the project root in VS Code, or launch from its notebooks/ directory.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent) if (p / 'src/data/cub_data.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the extracted project folder in VS Code before running this notebook.')
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'outputs/member1_eda'; OUT.mkdir(parents=True, exist_ok=True)
AUDIT = OUT / 'final_audit'; AUDIT.mkdir(exist_ok=True)
from src.data.member1_pipeline import (scan_images, image_flags, contact_sheets,
    captions_and_flags, make_manifest, check_exact_duplicates, verify_loaders, save_review, save_json)
from src.data.cub_data import make_classification_loaders, get_transforms

## 1. Load the original dataset
The official test split remains separate from model selection.

In [ ]:
train = load_dataset('Multimodal-Fatima/CUB_train', split='train')
test = load_dataset('Multimodal-Fatima/CUB_test', split='test')
print('Original train:', len(train), 'Official test:', len(test))
print('Species:', len(set(train['label'])))

## 2. EDA and image-quality review
Scan all original training images, chart class frequencies, save metadata and generate contact sheets for suspicious images. **Flags are not automatic exclusions.**

In [ ]:
counts = pd.Series(train['label']).value_counts().sort_index()
counts.rename_axis('label').reset_index(name='train_count').to_csv(OUT / '02_class_counts.csv', index=False)
counts.plot(kind='bar', figsize=(17,3), title='Training images per species', xticks=[])
plt.tight_layout(); plt.savefig(OUT/'02_class_distribution.png');plt.show()
train_meta = scan_images(train)
train_meta.to_csv(OUT/'03_train_image_metadata.csv', index=False)
flags = image_flags(train_meta)
flags.to_csv(AUDIT/'flagged_images_for_review.csv', index=False)
contact_sheets(train, flags, AUDIT)
print('Train image decode failures:', (~train_meta.decode_ok).sum(), 'Flagged images:',len(flags))

## 3. Caption cleaning and documented review
Normalize whitespace, remove identical per-image captions and the incomplete caption `this`. Preserve original source data. Contact-sheet review decisions are stored separately; new flags default to **pending**, never automatically `keep`.

In [ ]:
caption_report = captions_and_flags(train, OUT)
print('Caption cleaning:', caption_report)
print('Review image examples in', AUDIT, 'before marking them as kept.')
reviews = save_review(flags, AUDIT)
print('Image review decisions:', reviews['decision'].value_counts().to_dict())

## 4. Stratified training/validation manifest
Use an existing valid manifest when available, otherwise generate an 80/20 stratified split with seed 42.

In [ ]:
manifest = make_manifest(train, OUT)
print('Split sizes:', manifest['split'].value_counts().to_dict())
print('Species in each split:', manifest.groupby('split')['label'].nunique().to_dict())

## 5. Integrity checks and exact-duplicate leakage correction
Scan all official test images. Hash full decoded RGB pixels, repair same-class train/validation overlaps by swapping a class-matched training image, and stop if an exact duplicate overlaps the official test split.

In [ ]:
test_meta = scan_images(test, 'test')
test_meta.to_csv(AUDIT/'test_image_metadata.csv', index=False)
test_meta.loc[~test_meta.decode_ok].to_csv(AUDIT/'test_decode_failures.csv', index=False)
assert train_meta.decode_ok.all() and test_meta.decode_ok.all(), 'Review decoding failures before use'
identity_report, manifest = check_exact_duplicates(train, test, manifest, OUT)
print('Exact duplicate groups:', identity_report['exact_duplicate_groups'])
print('Cross-split duplicate groups after repair:', identity_report['cross_split_groups_after_fix'])

## 6. Preprocessing, augmentation and complete DataLoader checks
Training: random resized crop, flip, mild rotation and brightness/contrast jitter. Validation/test: deterministic resize and center crop. All inputs are RGB, 224×224, normalized. Test **every batch**.

In [ ]:
train_loader, val_loader, test_loader = make_classification_loaders(
    train, test, manifest_path=OUT/'train_validation_split.csv', batch_size=32, num_workers=0)
print('Train transforms:', get_transforms(train=True))
print('Evaluation transforms:', get_transforms(train=False))
results = verify_loaders((train_loader,val_loader,test_loader),
                         (int((manifest.split=='train').sum()),int((manifest.split=='validation').sum()),len(test)), AUDIT)
print('Full DataLoader verification:', results)

## 7. Save teammate handoff
Outputs are written to `outputs/member1_eda/`. Share this notebook with `src/data/*.py`, the corrected manifest, caption JSONL and review/audit reports.

In [ ]:
# Preserve the existing decisions and evidence in final output.
reviews = save_review(flags, AUDIT)
summary = {
    'original_train':len(train), 'official_test':len(test),
    'train_count':int((manifest.split=='train').sum()),
    'validation_count':int((manifest.split=='validation').sum()),
    'species':int(counts.size), 'image_size':224, 'batch_size':32,
    'flagged_images':len(flags),'image_review_decisions':reviews['decision'].value_counts().to_dict(),
    'caption_cleaning':caption_report,'duplicate_audit':identity_report,
    'full_loader_verification':results,
}
save_json(OUT/'handoff_summary.json',summary)
print('Saved final handoff summary:',OUT/'handoff_summary.json')